# Mediana dla obrazu kolorowego

Idea filtracji medianowej jest dość prosta dla obrazów w odcieniach szarości.
Dla obrazów kolorowych trudniej jest określić kryterium wg. którego szeregowane będą wartości, z których wyznaczana będzie mediana.

Jedną z możliwości wykonania filtracji medianowej dla obrazów kolorowych (na podstawie *The Image Processing Handbook*, J. Russ) jest wykorzystanie następującej definicji mediany:
``mediana to ten piksel z otoczenia, którego odległość do innych pikseli z otoczenia jest najmniejsza''.
Jako miarę odległości wykorzystujemy pierwiastek z sumy kwadratów różnic poszczególnych składowych R,G,B.
Zatem odległość między dwoma pikselami wyraża się wzorem:
\begin{equation}
dRGB = \sqrt{(R_1-R_2)^2+(G_1-G_2)^2+(B_1-B_2)^2}
\end{equation}

Warto zwrócić uwagę, że istnieje wiele możliwości zdefiniowania porównywania wielkości wektorowych (jeden piksel to wektor o trzech składowych).
Można zamiast odległości wykorzystać kąt albo połączyć oba parametry.
Ponadto istnieje możliwość dodania do wektora dodatkowych składowych - tak aby lepiej opisać piksel.

Celem zadania jest implementacja opisanego algorytmu.

1. Wczytaj obraz *lenaRGBSzum.png* (dostępny na git).
2. Zdefiniuj rozmiar okna.
3. Wykonaj pętle po pikselach, dla których okno jest zdefiniowane (pomiń brzeg obrazu).
4. Dla każdego piksela pobierz okno o właściwym rozmiarze.
5. Wykonaj pętle po oknie, wewnątrz której obliczona zostanie suma odległości.
    - Obliczanie różnicy: `window - window[rowWin, colWin]`.
    - Obliczanie kwadratów: `np.square`.
    - Obliczanie pierwiastka: `np.sqrt`.
    - Obliczanie sumy metodą `.sum`.
6. Po obliczeniu macierzy odległości wyznacz argument elementu minimalnego.
Wykorzystaj funkcję `np.argmin`.
Argument funkcji zostanie spłaszczony, jeśli ma więcej niż jeden wymiar.
Aby przekonwertować spłaszczony indeks na indeks macierzy wykorzystaj funkcję `np.unravel_index`.
7. Przypisz odpowiedni wektor wartości do piksela obrazu wynikowego.
8. Wyświetl obraz oryginalny i przefiltrowany.
9. Przeprowadź dwa eksperymenty - dla obrazu _lenaRGB_ oraz _lenaRGBszum_.

In [ ]:
import cv2
import os
import requests
from matplotlib import pyplot as plt
import numpy as np
from scipy import signal

url = 'https://raw.githubusercontent.com/vision-agh/poc_sw/master/06_Context/'

fileNames = ["lenaRGB.png", "lenaRGBSzum.png"]
for fileName in fileNames:
  if not os.path.exists(fileName):
      r = requests.get(url + fileName, allow_redirects=True)
      open(fileName, 'wb').write(r.content)

In [ ]:
def color_median_filter(image, window_size=3):
    half_window = window_size // 2
    rows, cols, channels = image.shape
    
    output_image = np.zeros_like(image)
    img_float = image.astype(np.float32)
    for i in range(half_window, rows - half_window):
        for j in range(half_window, cols - half_window):
            window = img_float[i-half_window : i+half_window+1, j-half_window : j+half_window+1]
            pixels = window.reshape(-1, 3)
            num_pixels = pixels.shape[0]
            distances_sum = np.zeros(num_pixels)
            for k in range(num_pixels):
                p1 = pixels[k]
                diff = pixels - p1
                dist_sq = np.sum(np.square(diff), axis=1)
                dist = np.sqrt(dist_sq)
                distances_sum[k] = np.sum(dist)
            min_dist_idx = np.argmin(distances_sum)
            output_image[i, j] = pixels[min_dist_idx]

    return output_image

lena_szum = cv2.imread("lenaRGBSzum.png")
lena_szum = cv2.cvtColor(lena_szum, cv2.COLOR_BGR2RGB)

lena_clean = cv2.imread("lenaRGB.png")
lena_clean = cv2.cvtColor(lena_clean, cv2.COLOR_BGR2RGB)

filtered_szum = color_median_filter(lena_szum, window_size=3)
filtered_clean = color_median_filter(lena_clean, window_size=3)

fig, ax = plt.subplots(2, 2, figsize=(12, 12))

ax[0][0].imshow(lena_szum)
ax[0][0].set_title("Lena Szum Oryginał")
ax[0][0].axis('off')

ax[0][1].imshow(filtered_szum)
ax[0][1].set_title("Lena Szum po filtracji medianowej")
ax[0][1].axis('off')

ax[1][0].imshow(lena_clean)
ax[1][0].set_title("Lena Oryginał")
ax[1][0].axis('off')

ax[1][1].imshow(filtered_clean)
ax[1][1].set_title("Lena po filtracji medianowej")
ax[1][1].axis('off')

plt.tight_layout()
plt.show()